# Unidad 2.2 - LlamaIndex

**LlamaIndex** es un toolkit completo para crear agentes impulsados por LLMs sobre tus datos, usando índices y flujos de trabajo (workflows).

**Temas cubiertos:**
1. **Componentes** — Pipeline RAG: carga, indexación, almacenamiento, consulta y evaluación
2. **Workflows** — Arquitectura orientada a eventos para flujos de control flexibles
3. **Multi-Agent Workflows** — Sistema de múltiples agentes colaborando

> Alfred necesita buscar, sintetizar y recuperar información de múltiples fuentes para organizar la fiesta perfecta. 🦇

## Instalación

In [1]:
%pip install llama-index
%pip install llama-index-embeddings-huggingface
%pip install llama-index-llms-huggingface-api
%pip install llama-index-vector-stores-chroma
%pip install llama-index-utils-workflow
%pip install chromadb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached llama_index_embeddings_huggingface-0.7.0-py3-none-any.whl.metadata (459 bytes)
  Using cached sentence_transformers-5.5.0-py3-none-any.whl.metadata (18 kB)
Using cached llama_index_embeddings_huggingface-0.7.0-py3-none-any.whl (8.9 kB)
Using cached sentence_transformers-5.5.0-py3-none-any.whl (588 kB)

   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ---------------------------------------- 0/2 [sentence-transformers]
   ------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


...

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 756.0/756.0 kB 14.3 MB/s  0:00:00

   ---------------------------------------- 0/3 [jsonpickle]
   ------------- -------------------------- 1/3 [pyvis]
   ---------------------------------------- 3/3 [llama-index-utils-workflow]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Parte 1: Componentes — Pipeline RAG con QueryEngine

El componente **QueryEngine** es la pieza clave para construir flujos de trabajo RAG en LlamaIndex. Permite al agente encontrar y utilizar información relevante de tus datos.

### Las 5 etapas de un pipeline RAG:
1. **Loading** — Obtener datos desde archivos, PDFs, bases de datos, APIs
2. **Indexing** — Crear embeddings vectoriales de los datos
3. **Storing** — Persistir el índice para evitar re-indexación
4. **Querying** — Consultar el índice con LLMs
5. **Evaluation** — Medir la calidad de las respuestas

### Paso 1: Carga y embedding de documentos

In [2]:
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# Documentos de ejemplo sobre la fiesta de Wayne Manor
documents = [
    Document(text="La mansión Wayne es famosa por sus lujosas fiestas con decoraciones temáticas de superhéroes."),
    Document(text="Alfred es el mayordomo de confianza de Bruce Wayne, experto en organización de eventos de alta gama."),
    Document(text="Los mejores catering de Gotham City incluyen Gotham Catering Co., Wayne Manor Catering y Gotham City Events."),
    Document(text="Para una fiesta de máscaras de villanos se recomiendan decoraciones oscuras, música electrónica y cocktails temáticos."),
    Document(text="El Chef Gordon Ramsay recomienda servir filet mignon y langosta para cenas formales de superhéroes."),
]

# Pipeline de ingesta con transformaciones
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=0),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ]
)

nodes = await pipeline.arun(documents=documents)
print(f"Documentos procesados en {len(nodes)} nodos")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\USUARIO\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USUARIO\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documentos procesados en 5 nodos


### Paso 2: Almacenamiento e indexación con ChromaDB

In [3]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

# Crear base de datos Chroma persistente
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Pipeline con almacenamiento vectorial integrado
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=0),
        embed_model,
    ],
    vector_store=vector_store,
)

await pipeline.arun(documents=documents)
print("Documentos indexados y almacenados en ChromaDB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Documentos indexados y almacenados en ChromaDB


### Paso 3: Indexar desde el vector store

In [4]:
from llama_index.core import VectorStoreIndex

# Crear índice desde el vector store existente
index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
print("Índice cargado desde ChromaDB")

Índice cargado desde ChromaDB


### Paso 4: Consultar el índice con un LLM

In [5]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

# Opciones de conversión del índice:
# - as_retriever: recuperación básica de documentos
# - as_query_engine: pregunta-respuesta en una sola interacción
# - as_chat_engine: conversación que mantiene historial
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",  # compact, refine, tree_summarize
)

response = query_engine.query("¿Cuáles son los mejores servicios de catering en Gotham City?")
print(response)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d1256-27974b463524ae72196a0a32;c5abedd0-bfba-4295-a387-bb38563d1f50)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Paso 5: Evaluación de respuestas

In [6]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# Evaluar si la respuesta está respaldada por el contexto recuperado
evaluator = FaithfulnessEvaluator(llm=llm)
response = query_engine.query("¿Qué recomienda Alfred para una fiesta formal?")
eval_result = evaluator.evaluate_response(response=response)

print(f"¿La respuesta es fiel al contexto? {eval_result.passing}")
print(f"Respuesta: {response}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d1257-7c7b7a5f79d9fc613305e96d;be43c030-0b1c-469f-ab9f-c5e6327f00fe)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

---
## Parte 2: Workflows

Un **Workflow** en LlamaIndex proporciona una forma estructurada de organizar el código en pasos secuenciales y manejables.

Los workflows se crean definiendo `Steps` que son disparados por `Events`, y a su vez emiten `Events` para disparar pasos posteriores.

**Ventajas:**
- Organización clara del código en pasos discretos
- Arquitectura orientada a eventos para flujo de control flexible
- Comunicación con tipos seguros entre pasos
- Gestión de estado integrada

### Workflow básico de un solo paso

In [7]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step

class AlfredWorkflow(Workflow):
    @step
    async def greet(self, ev: StartEvent) -> StopEvent:
        return StopEvent(result="¡Bienvenido a Wayne Manor! ¿En qué puedo servirle?")

w = AlfredWorkflow(timeout=10, verbose=False)
result = await w.run()
print(result)

¡Bienvenido a Wayne Manor! ¿En qué puedo servirle?


### Workflow de múltiples pasos conectados

Para conectar múltiples pasos, creamos **eventos personalizados** que llevan datos entre pasos.

In [8]:
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step

# Evento intermedio que transfiere datos entre pasos
class PartyPlanEvent(Event):
    task_description: str

class PartyWorkflow(Workflow):
    @step
    async def plan_party(self, ev: StartEvent) -> PartyPlanEvent:
        print("Alfred: Analizando los requisitos de la fiesta...")
        return PartyPlanEvent(task_description="Fiesta de máscaras de villanos en Wayne Manor")

    @step
    async def prepare_details(self, ev: PartyPlanEvent) -> StopEvent:
        print(f"Alfred: Preparando detalles para: {ev.task_description}")
        result = f"Plan completado: {ev.task_description} — decoraciones oscuras, música electrónica, catering de lujo."
        return StopEvent(result=result)

w = PartyWorkflow(timeout=10, verbose=True)
result = await w.run()
print(f"\nResultado final: {result}")

[tick] add: StartEvent()
[plan_party:0] started from StartEvent
Alfred: Analizando los requisitos de la fiesta...
[plan_party:0] complete with PartyPlanEvent
[tick] add: PartyPlanEvent(task_description='Fiesta de máscaras de villanos en Wayne Manor')
[prepare_details:0] started from PartyPlanEvent
Alfred: Preparando detalles para: Fiesta de máscaras de villanos en Wayne Manor
[result] StopEvent(result='Plan completado: Fiesta de máscaras de villanos en...')
[prepare_details:0] complete with StopEvent

Resultado final: Plan completado: Fiesta de máscaras de villanos en Wayne Manor — decoraciones oscuras, música electrónica, catering de lujo.


### Workflows con bucles y ramas

El **tipado** es la parte más poderosa de los workflows — permite crear ramas, bucles y joins para flujos complejos.

In [9]:
from llama_index.core.workflow import Event, StartEvent, StopEvent, Workflow, step
import random

class TaskCompletedEvent(Event):
    result: str

class RetryEvent(Event):
    reason: str

class RobustPartyWorkflow(Workflow):
    @step
    async def attempt_task(self, ev: StartEvent | RetryEvent) -> TaskCompletedEvent | RetryEvent:
        # Simula un proceso que puede fallar aleatoriamente
        if random.randint(0, 1) == 0:
            print("Alfred: Hubo un problema, reintentando...")
            return RetryEvent(reason="El proveedor de catering no respondió")
        else:
            print("Alfred: ¡Tarea completada exitosamente!")
            return TaskCompletedEvent(result="Catering confirmado para Wayne Manor")

    @step
    async def finalize(self, ev: TaskCompletedEvent) -> StopEvent:
        return StopEvent(result=f"Proceso finalizado: {ev.result}")

w = RobustPartyWorkflow(verbose=False)
result = await w.run()
print(f"Resultado: {result}")

Alfred: ¡Tarea completada exitosamente!
Resultado: Proceso finalizado: Catering confirmado para Wayne Manor


### Gestión de estado con Context

El `Context` permite que todos los pasos del workflow accedan al mismo estado compartido.

In [10]:
from llama_index.core.workflow import Context, Event, StartEvent, StopEvent, Workflow, step

class GuestAddedEvent(Event):
    guest_name: str

class GuestListWorkflow(Workflow):
    @step
    async def add_guests(self, ctx: Context, ev: StartEvent) -> GuestAddedEvent:
        # Guardar el contexto inicial
        await ctx.store.set("guest_count", 0)
        await ctx.store.set("guests", [])
        return GuestAddedEvent(guest_name="Bruce Wayne")

    @step
    async def record_guest(self, ctx: Context, ev: GuestAddedEvent) -> StopEvent:
        # Recuperar y actualizar el estado
        guests = await ctx.store.get("guests")
        guests.append(ev.guest_name)
        count = len(guests)
        await ctx.store.set("guests", guests)
        await ctx.store.set("guest_count", count)

        return StopEvent(result=f"Lista de invitados actualizada. Total: {count} invitado(s): {guests}")

w = GuestListWorkflow(timeout=10, verbose=False)
result = await w.run()
print(result)

Lista de invitados actualizada. Total: 1 invitado(s): ['Bruce Wayne']


---
## Parte 3: Multi-Agent Workflows con AgentWorkflow

En lugar de crear workflows manualmente, podemos usar la clase **`AgentWorkflow`** para crear un sistema de múltiples agentes que colaboran y se transfieren tareas según sus capacidades especializadas.

In [11]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

# Definir herramientas para los agentes
def calculate_budget(items: int, cost_per_item: float) -> float:
    """Calcula el presupuesto total para la fiesta."""
    return items * cost_per_item

def count_guests(guest_list: str) -> int:
    """Cuenta el número de invitados en una lista separada por comas."""
    return len([g.strip() for g in guest_list.split(',') if g.strip()])

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

# Agente especializado en presupuesto
budget_agent = ReActAgent(
    name="budget_agent",
    description="Especialista en cálculos de presupuesto para eventos",
    system_prompt="Eres Alfred, el mayordomo de Wayne Manor, experto en gestión de presupuestos para eventos de lujo.",
    tools=[calculate_budget],
    llm=llm,
)

# Agente especializado en gestión de invitados
guest_agent = ReActAgent(
    name="guest_agent",
    description="Especialista en gestión de invitados y listas de asistencia",
    system_prompt="Eres Alfred, el mayordomo de Wayne Manor, experto en gestión de invitados para eventos de alta gama.",
    tools=[count_guests],
    llm=llm,
)

# Crear el workflow multi-agente
workflow = AgentWorkflow(
    agents=[budget_agent, guest_agent],
    root_agent="budget_agent",
)

# Ejecutar el sistema
response = await workflow.run(
    user_msg="¿Cuántos invitados hay en esta lista: 'Bruce Wayne, Clark Kent, Diana Prince, Barry Allen'?"
)
print(response)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d1258-339e32c703ba0886207f877c;18c9cc83-9e23-462d-87a2-7497f3adc417)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### Multi-Agent Workflow con estado compartido

Los agentes pueden modificar el estado del workflow usando `Context`. El estado inicial se proporciona antes de iniciar el workflow.

In [12]:
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

# Herramientas que actualizan el estado compartido
async def add_to_shopping_list(ctx: Context, item: str) -> str:
    """Agrega un artículo a la lista de compras de la fiesta."""
    cur_state = await ctx.store.get("state")
    cur_state["shopping_list"].append(item)
    cur_state["num_items"] += 1
    await ctx.store.set("state", cur_state)
    return f"'{item}' añadido a la lista. Total: {cur_state['num_items']} artículos."

async def get_shopping_list(ctx: Context) -> str:
    """Obtiene la lista de compras actual de la fiesta."""
    cur_state = await ctx.store.get("state")
    items = cur_state.get("shopping_list", [])
    return f"Lista de compras ({len(items)} artículos): {', '.join(items) if items else 'vacía'}"

shopping_agent = ReActAgent(
    name="shopping_agent",
    description="Gestiona la lista de compras para la fiesta de Wayne Manor",
    system_prompt="Eres Alfred, el mayordomo. Gestiona meticulosamente la lista de compras para la fiesta.",
    tools=[add_to_shopping_list, get_shopping_list],
    llm=llm,
)

workflow = AgentWorkflow(
    agents=[shopping_agent],
    root_agent="shopping_agent",
    initial_state={"shopping_list": [], "num_items": 0},
    state_prompt="Estado actual: {state}. Mensaje del usuario: {msg}",
)

ctx = Context(workflow)

# Agregar artículos a la lista
response = await workflow.run(user_msg="Agrega champagne y caviar a la lista de compras", ctx=ctx)
print(response)

# Verificar el estado
state = await ctx.store.get("state")
print(f"\nEstado final: {state}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a0d1259-06440b5530cb84ec0f871fa5;bdf87609-9d6e-40cb-b221-9dc0b8f93fbb)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

---
## Resumen

| Concepto | Descripción |
|---|---|
| **SimpleDirectoryReader** | Carga documentos desde un directorio local |
| **IngestionPipeline** | Procesa y transforma documentos en nodos con embeddings |
| **VectorStoreIndex** | Índice vectorial para búsqueda semántica |
| **QueryEngine** | Interfaz de pregunta-respuesta sobre el índice |
| **Workflow** | Arquitectura orientada a eventos para flujos de agentes |
| **AgentWorkflow** | Sistema multi-agente con transferencia de tareas |
| **Context** | Estado compartido accesible por todos los pasos |

**Recursos:**
- [Documentación oficial de LlamaIndex](https://docs.llamaindex.ai/)
- [LlamaHub — registry de componentes](https://llamahub.ai/)
- [Guía de Workflows](https://docs.llamaindex.ai/en/stable/understanding/workflows/)